In [2]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.sql.functions import col
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pandas as pd
from sklearn.metrics import classification_report

In [3]:

# 1. Spark
spark = SparkSession.builder.appName("SentimentAnalysis").getOrCreate()

# 2. Đọc dữ liệu
df = spark.read.csv("/content/sentiments.csv", header=True, inferSchema=True)
df = df.withColumn("label", (col("sentiment").cast("integer") + 1) / 2)
df = df.withColumn("label", col("label").cast("integer"))
df = df.dropna(subset=["sentiment"])
print(f"Loaded {df.count()} rows")

# 3. Chia tập
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

# 4. Pipeline
pipeline = Pipeline(stages=[
    Tokenizer(inputCol="text", outputCol="words"),
    StopWordsRemover(inputCol="words", outputCol="filtered_words"),
    HashingTF(inputCol="filtered_words", outputCol="raw_features", numFeatures=10000),
    IDF(inputCol="raw_features", outputCol="features"),
    LogisticRegression(maxIter=10, regParam=0.001)
])

# 5. Train & Predict
model = pipeline.fit(train_data)
predictions = model.transform(test_data)

# 6. Accuracy
accuracy = MulticlassClassificationEvaluator(metricName="accuracy").evaluate(predictions)
print(f"\nAccuracy: {accuracy:.4f}")

# 7. Classification Report
pred_pd = predictions.select("label", "prediction").toPandas()
report = classification_report(pred_pd["label"], pred_pd["prediction"].astype(int),
                              target_names=['negative', 'positive'], digits=2, output_dict=True)
print("\nClassification Report:")
print(pd.DataFrame(report).transpose().round(4))

spark.stop()

Loaded 5791 rows

Accuracy: 0.7295

Classification Report:
              precision  recall  f1-score    support
negative         0.6534  0.5938    0.6222   416.0000
positive         0.7688  0.8110    0.7893   693.0000
accuracy         0.7295  0.7295    0.7295     0.7295
macro avg        0.7111  0.7024    0.7057  1109.0000
weighted avg     0.7255  0.7295    0.7266  1109.0000


In [ ]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from src.models.text_classifier import TextClassifier

# === HÀM LÀM SẠCH VĂN BẢN ===
def clean_text(text):
    # 1. Chuyển về lowercase
    text = text.lower()

    # 2. Loại bỏ @user, #hashtag, http links
    text = re.sub(r'@\w+|#\w+|http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # 3. Loại bỏ ký tự không phải chữ/số (giữ khoảng trắng)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)

    # 4. Thay nhiều khoảng trắng bằng 1
    text = re.sub(r'\s+', ' ', text).strip()

    # 5. Loại bỏ từ ngắn vô nghĩa (<2 ký tự) hoặc mã chứng khoán kiểu "XIDE"
    words = text.split()
    words = [w for w in words if len(w) >= 2 and not (w.isupper() and len(w) <= 5)]
    return ' '.join(words)

# === ĐỌC DỮ LIỆU ===
df = pd.read_csv("sentiments.csv")
df["text_clean"] = df["text"].astype(str).apply(clean_text)

print("Before cleaning:")
print(df["text"].iloc[0])
print("\nAfter cleaning:")
print(df["text_clean"].iloc[0])